In [2]:
import nltk
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
import nltk
import string
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [3]:
def preprocess_text(text):
    # Preprocess the text
    tokens = word_tokenize(text)  # Word tokenization
    processed_tokens = []

    for token in tokens:  # Lowercasing
        processed_tokens.append(token.lower())

    stop_words = set(stopwords.words('english'))  # Remove stopwords
    filtered_tokens = []

    for word in processed_tokens:
        # Keep words that are not in stop words OR are single characters
        # (to potentially keep stemmed words)
        if word not in stop_words or len(word) <= 1:
            filtered_tokens.append(word)

    stemmer = PorterStemmer()  # Stemming: bring words back to base form
    stemmed_tokens = []

    for word in filtered_tokens:
        stemmed_tokens.append(stemmer.stem(word))

    # Remove any remaining single-character tokens that might be punctuation
    final_tokens = [token for token in stemmed_tokens if len(token) > 1 or token.isalpha()]

    return " ".join(final_tokens)  # Tokens separated by space

In [4]:
documents=[
  "The quick brown fox jumps over the lazy dog.",
  "Never jump over the lazy dog quickly.",
  "A fast brown fox leaps across lazy dogs in summer.",
  "Cats and dogs are common pets.",
  "The fox is a wild animal."
]

preprocessed_docs = [preprocess_text(doc) for doc in documents]

In [5]:
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(preprocessed_docs)
vocab = vectorizer.get_feature_names_out()

In [6]:
def process_query(query):
    processed_query = preprocess_text(query)  # Preprocessing
    query_vector = vectorizer.transform([processed_query])  # Transform
    return query_vector


def get_similarity_score(query_vector):
    similarity = cosine_similarity(query_vector, tfidf_matrix)
    return similarity.flatten()  # Convert 2-D to 1-D matrix


def retrieve_documents(query, documents):
    query_vector = process_query(query)  # Call function for processing query
    scores = get_similarity_score(query_vector)  # Function for cosine similarity
    ranked_indices = np.argsort(scores)[::-1]  # Descending order of scores

    # Remove documents with 0 or negative score
    ranked_docs = [(documents[i], scores[i]) for i in ranked_indices if scores[i] > 0]

    return ranked_docs

In [7]:
test_query="fox and dog"
results=retrieve_documents(test_query, documents)
print(f"Query: '{test_query}' \n")
print("Ranked Documents: \n")
for i, (doc, score) in enumerate(results, 1): #countrer will start from 1
  print(f"{i}. Score: {score: .4f} | Document: {doc}")

Query: 'fox and dog' 

Ranked Documents: 

1. Score:  0.4667 | Document: The quick brown fox jumps over the lazy dog.
2. Score:  0.3614 | Document: A fast brown fox leaps across lazy dogs in summer.
3. Score:  0.3275 | Document: The fox is a wild animal.
4. Score:  0.1991 | Document: Cats and dogs are common pets.
5. Score:  0.1962 | Document: Never jump over the lazy dog quickly.


In [8]:
print(preprocessed_docs)

['quick brown fox jump lazi dog', 'never jump lazi dog quickli', 'a fast brown fox leap across lazi dog summer', 'cat dog common pet', 'fox a wild anim']


Q. Discuss the impact of preprocessing steps and TF-IDF transformation on the retrieval accu-
racy.

Impact of Preprocessing Steps and TF-IDF Transformation on Retrieval Accuracy
1. Preprocessing Steps Impact:
- Text cleaning: Converts words to lowercase, removes stop words, and stems words to base forms
- Better matching: Helps find relevant documents even when words are written differently
2. TF-IDF Transformation Impact:
- Smart weighting: Gives more importance to rare, meaningful words and less to common words
- Better ranking: Documents with unique matching terms rank higher than those with only common words
3. Overall Impact:
- Higher accuracy: Both steps together reduce noise and improve document ranking
- More relevant results: Users get documents that truly match their search intent